In [ ]:
import sys
import glob
import os.path as osp
from tqdm import tqdm

sys.path.append("../")

from src.client.core.carla_annotator import transfer_scenario_to_video_ultralytics

# %load_ext autoreload
# %autoreload 2


## Generate videos from generated images
- Greatly reducing the required space.
- Create a new dir, copy the rest of files.

In [ ]:
# frame_rate = 20
image_format = "png"
output_dir = "/home/marek/Work/CarlaPOC/runs/export"

scenario_dirs = glob.glob(osp.join(output_dir, "*"))
for scenario_dir in tqdm(scenario_dirs, total=len(scenario_dirs)):
    if scenario_dir.endswith("_video") or "cropped" in scenario_dir:  # Do not search dirs with generated videos
        continue
    if not osp.isdir(scenario_dir):
        continue
    transfer_scenario_to_video_ultralytics(
        scenario_dir=scenario_dir,
        frame_rate=None,
        image_format=image_format,
    )


# Crop videos

In [ ]:
import subprocess
import cv2


def crop_video_ffmpeg(input_path: str, output_path: str, fps: float, start_frame: int, end_frame: int, reencode: bool = False) -> None:
    """
    Crop a video between two frame numbers using ffmpeg.

    """
    start_time = start_frame / fps
    duration = (end_frame - start_frame) / fps

    if reencode:
        cmd = [
            "ffmpeg",
            "-y",
            "-i", input_path,
            "-ss", str(start_time),
            "-t", str(duration),
            "-c:v", "libx264",
            "-an",                 # disable audio
            output_path,
        ]
    else:
        cmd = [
            "ffmpeg",
            "-y",
            "-ss", str(start_time),
            "-t", str(duration),
            "-i", input_path,
            "-c", "copy",
            output_path,
        ]

    result = subprocess.run(cmd, capture_output=True, text=True)
    print("STDOUT:\n", result.stdout)
    print("STDERR:\n", result.stderr)
    result.check_returncode()

    print(f"Cropped video saved to {output_path}")


In [ ]:
import os
import os.path as osp
import numpy as np
from src.client.core.ioutils import load_yaml, load_json, save_json
import shutil
from pathlib import Path


def get_random_start_frame(start_frame: int, collision_frame: int, fps: float, mean: float = 5.0, sigma: float = 2.5, min_diff_secs: float = 3) -> int:
    min_diff_frames = int(min_diff_secs * fps)
    max_diff_frames = collision_frame - start_frame

    if max_diff_frames <= min_diff_frames:
        return start_frame

    offset_secs = np.random.normal(mean, sigma)
    offset_secs = max(offset_secs, min_diff_frames / fps)
    offset_secs = min(offset_secs, max_diff_frames / fps)

    offset_frames = int(round(offset_secs * fps))
    start_frame_random = collision_frame - offset_frames

    return max(int(start_frame_random), start_frame)


def get_random_end_frame(end_frame: int, collision_frame: int, fps: float, mean: float = 3.0, sigma: float = 2.5, min_diff_secs: float = 3) -> int:
    min_diff_frames = min_diff_secs * fps
    crop_time = np.random.normal(mean, sigma)
    crop_frames = crop_time * fps
    end_frame_random = max(end_frame - crop_frames, collision_frame + min_diff_frames)
    end_frame_random = min(end_frame_random, end_frame)
    return int(end_frame_random)


export_dir = "../runs/export_20250916"
scenario_names = os.listdir(export_dir)
for scenario_name in tqdm(scenario_names, total=len(scenario_names)):
    video_dir = osp.join(export_dir, scenario_name)
    cropped_dir = osp.join(export_dir + "_cropped", scenario_name)

    if not osp.exists(cropped_dir):
        os.makedirs(cropped_dir)

    exp_names = os.listdir(video_dir)
    for exp_name in exp_names:
        exp_dir = osp.join(video_dir, exp_name)
        exp_cropped_dir = osp.join(cropped_dir, exp_name)
        if not osp.isdir(exp_dir):
            print(f"{exp_name} does not exist")
            continue
        if not osp.isdir(exp_cropped_dir):
            os.makedirs(exp_cropped_dir)

        video_image_dir = osp.join(exp_dir, "images", "train")
        video_annotation_dir = osp.join(exp_dir, "labels", "train")
        cropped_image_dir = osp.join(exp_cropped_dir, "images", "train")
        cropped_annotation_dir = osp.join(exp_cropped_dir, "labels", "train")
        os.makedirs(cropped_image_dir, exist_ok=True)
        os.makedirs(cropped_annotation_dir, exist_ok=True)

        config = load_yaml(osp.join(exp_dir, f"{osp.basename(video_dir).replace('_video', '')}.yaml"))
        frame_rate = (
                config["template"]["simulation_fps"]
                / config["template"]["frames_per_image"]
        )
        display_size = config["template"]["display_size"]


        for label_file in os.listdir(video_annotation_dir):
            # labels
            label_path = osp.join(video_annotation_dir, label_file)
            metadata = load_json(label_path)
            base, collision, sensor  = metadata["base"], metadata["collision"], metadata["sensor"]
            first_frame, last_frame = base[0]["iteration"], base[-1]["iteration"]
            collision_start_frame = int(collision[0]["iteration"])

            start_frame = get_random_start_frame(first_frame, collision_start_frame, frame_rate)
            end_frame = get_random_end_frame(last_frame, collision_start_frame, frame_rate, mean=1, sigma=1)

            base = [frame_data for frame_data in base if start_frame <= frame_data["iteration"] <= end_frame]
            collision = [frame_data for frame_data in collision if frame_data["iteration"] <= end_frame]

            cropped_metadata = {
                "base": base,
                "collision": collision,
                "sensor": sensor,
            }
            save_json(path=osp.join(cropped_annotation_dir, label_file), content=cropped_metadata)

            # videos
            timestamp = Path(label_file).stem
            video_files = [video_file for video_file in os.listdir(video_image_dir) if timestamp in video_file]

            for video_file in video_files:
                video_path = osp.join(video_image_dir, video_file)
                crop_video_ffmpeg(
                    input_path=video_path,
                    output_path=osp.join(cropped_image_dir, video_file),
                    fps=frame_rate,
                    start_frame=start_frame-first_frame,
                    end_frame=end_frame,
                    reencode=True
                )

        for config in glob.glob(osp.join(exp_dir, "*.yaml")):
            shutil.copy(
                config,
                osp.join(exp_cropped_dir, osp.basename(config)),
            )



## Crop collisions

In [ ]:
def extract_video_frame(video_path: str, save_path: str, frame_number: int) -> None:
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {video_path}")

    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)
    success, frame = cap.read()
    cap.release()

    if not success:
        raise RuntimeError(f"Failed to read frame {frame_number} from {video_path}")

    os.makedirs(osp.dirname(save_path), exist_ok=True)
    cv2.imwrite(save_path, frame)
    print(f"Saved collision frame {frame_number} from {video_path} -> {save_path}")


export_dir = "../runs/export_20250916_cropped"
scenario_names = os.listdir(export_dir)
for scenario_name in tqdm(scenario_names, total=len(scenario_names)):
    video_dir = osp.join(export_dir, scenario_name)
    cropped_dir = osp.join(export_dir + "_collisions_only", scenario_name)

    if not osp.exists(cropped_dir):
        os.makedirs(cropped_dir)

    exp_names = os.listdir(video_dir)
    for exp_name in exp_names:
        exp_dir = osp.join(video_dir, exp_name)
        exp_cropped_dir = osp.join(cropped_dir, exp_name)
        if not osp.isdir(exp_dir):
            print(f"{exp_name} does not exist")
            continue
        if not osp.isdir(exp_cropped_dir):
            os.makedirs(exp_cropped_dir)

        video_image_dir = osp.join(exp_dir, "images", "train")
        video_annotation_dir = osp.join(exp_dir, "labels", "train")
        cropped_image_dir = osp.join(exp_cropped_dir, "images", "train")
        cropped_annotation_dir = osp.join(exp_cropped_dir, "labels", "train")
        os.makedirs(cropped_image_dir, exist_ok=True)
        os.makedirs(cropped_annotation_dir, exist_ok=True)

        config = load_yaml(osp.join(exp_dir, f"{osp.basename(video_dir).replace('_video', '')}.yaml"))
        frame_rate = (
                config["template"]["simulation_fps"]
                / config["template"]["frames_per_image"]
        )
        display_size = config["template"]["display_size"]


        for label_file in os.listdir(video_annotation_dir):
            # labels
            label_path = osp.join(video_annotation_dir, label_file)
            metadata = load_json(label_path)
            base, collision, sensor  = metadata["base"], metadata["collision"], metadata["sensor"]
            first_frame, last_frame = base[0]["iteration"], base[-1]["iteration"]
            collision_start_frame = int(collision[0]["iteration"])

            base = [frame_data for frame_data in base if frame_data["iteration"] == collision_start_frame]
            collision = collision[0]

            cropped_metadata = {
                "base": base,
                "collision": collision,
                "sensor": sensor,
            }
            save_json(path=osp.join(cropped_annotation_dir, label_file), content=cropped_metadata)

            # videos
            timestamp = Path(label_file).stem
            video_files = [video_file for video_file in os.listdir(video_image_dir) if timestamp in video_file]

            for video_file in video_files:
                video_path = osp.join(video_image_dir, video_file)
                save_path = osp.join(cropped_image_dir, video_file.replace(".mp4", f".jpg"))
                extract_video_frame(
                    video_path=video_path,
                    save_path=save_path,
                    frame_number=collision_start_frame - first_frame  # convert to relative frame index
                )
        for config in glob.glob(osp.join(exp_dir, "*.yaml")):
            shutil.copy(
                config,
                osp.join(exp_cropped_dir, osp.basename(config)),
            )
